## Expert Knowledge Worker

### A question answering agent that is an expert knowledge worker
### To be used by employees of Insurellm, an Insurance Tech company
### The agent needs to be accurate and the solution should be low cost.

This project will use RAG (Retrieval Augmented Generation) to ensure our question/answering assistant has high accuracy.

## TODAY:

- Part A: We will divide our documents into CHUNKS
- Part B: We will encode our CHUNKS into VECTORS and put in Chroma
- Part C: We will visualize our vectors

### PART A: Divide our documents into chunks

In [1]:
# pip install tiktoken langchain_openai langchain_chroma langchain_huggingface langchain_community langchain_text_splitters scikit-learn

In [3]:
pip install plotly

  Using cached plotly-6.7.0-py3-none-any.whl.metadata (8.6 kB)
Using cached plotly-6.7.0-py3-none-any.whl (9.9 MB)
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go

ModuleNotFoundError: No module named 'plotly'

In [5]:
# price is a factor for our company, so we're going to use a low cost model

MODEL = "llama-3.1-8b-instant"
db_name = "vector_db"
load_dotenv(override=True)
openai_api_key = os.getenv('GROQ_API_KEY')



In [6]:
# How many characters in all the documents?

knowledge_base_path = "knowledge-base/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base")

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")

Found 76 files in the knowledge base
Total characters in knowledge base: 304,434


In [8]:
# How many tokens in all the documents?


try:
    encoding = tiktoken.encoding_for_model(MODEL)
except KeyError:
    # Fallback for non-OpenAI models like Llama
    encoding = tiktoken.get_encoding("cl100k_base")

tokens = encoding.encode(entire_knowledge_base)
token_count = len(tokens)

print(f"Total tokens for {MODEL}: {token_count:,}")

Total tokens for llama-3.1-8b-instant: 63,721


In [9]:
# Load in everything in the knowledgebase using LangChain's loaders

folders = glob.glob("knowledge-base/*")

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

Loaded 76 documents


In [10]:
documents[1]

Document(metadata={'source': 'knowledge-base/contracts/Contract with GlobalRe Partners for Rellm.md', 'doc_type': 'contracts'}, page_content="# Contract with GlobalRe Partners for Rellm - AI-Powered Enterprise Reinsurance Solution\n\n**Contract Date:** April 28, 2025\n**Contract Number:** RE-2025-E-0203\n**Parties:**\n- Insurellm, Inc.\n- GlobalRe Partners International, Ltd.\n\n---\n\n## Terms\n\n1. **Coverage:** Insurellm agrees to provide GlobalRe Partners with enterprise access to the Rellm platform, including white-label capabilities, unlimited treaty and facultative reinsurance administration, dedicated infrastructure, and full API access for their global reinsurance operations.\n\n2. **Duration:** This agreement is effective for a period of 48 months from the contract date, representing Insurellm's most strategic long-term reinsurance partnership.\n\n3. **Payment:** GlobalRe Partners shall pay custom Enterprise Tier pricing of $45,000 per month for months 1-12, $48,000 per month

In [11]:
# Divide into chunks using the RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 413 chunks
First chunk:

page_content='# Contract with Pinnacle Insurance Co. for Homellm

## Terms
This contract ("Contract") is entered into as of this 1st day of January 2024 ("Effective Date") by and between Insurellm ("Provider"), a Delaware corporation with its principal place of business at 1234 Innovation Drive, San Francisco, CA 94105, and Pinnacle Insurance Co. ("Client"), a Texas corporation with its principal place of business at 4567 Protection Plaza, Houston, TX 77001. 

1. **License Grant**: Insurellm hereby grants the Client a non-exclusive, non-transferable license to use Homellm in accordance with the terms of this Contract.
2. **Payment Terms**: The Client agrees to pay an initial setup fee of $15,000 and a monthly subscription fee of $10,000 for the duration of the Contract.
3. **Term**: The initial term of this Contract shall last for a period of two (2) years from the Effective Date.' metadata={'source': 'knowledge-base/contracts/Contract with Pinnacl

In [12]:
chunks[100]

Document(metadata={'source': 'knowledge-base/contracts/Contract with Atlantic Risk Solutions for Bizllm.md', 'doc_type': 'contracts'}, page_content="---\n\n## Features\n\n1. **Access to Professional Tier Features**: Atlantic Risk Solutions will have access to all Professional Tier features, including:\n   - Multi-Line Underwriting Engine supporting general liability, professional liability, property, workers' compensation, and cyber insurance\n   - Business Intelligence Integration with automated data gathering from 50+ data sources\n   - Cyber Risk Assessment module with security posture evaluation\n   - Workers' Compensation Optimization with payroll integration and class code analysis\n   - Commercial Property Evaluation with catastrophe modeling\n   - Professional Liability Specialization for E&O coverage\n   - Portfolio Management Dashboard with loss ratio tracking and geographic analysis\n   - Agent and Broker Portal supporting up to 500 distribution partners\n   - Claims Managem

### PART B: Make vectors and store in Chroma

In Week 3, you set up a Hugging Face account and got an HF_TOKEN

At this point, you might want to add it to your `.env` file and run `load_dotenv(override=True)`

(This actually shouldn't be required).

In [13]:
# Pick an embedding model

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
#embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

ImportError: Could not import sentence_transformers python package. Please install it with `pip install sentence-transformers`.

In [ ]:
# Let's investigate the vectors

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

### Part C: Visualize!

In [ ]:
# Prework

result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['doc_type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [ ]:
# We humans find it easier to visalize things in 2D!
# Reduce the dimensionality of the vectors to 2D using t-SNE
# (t-distributed stochastic neighbor embedding)

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [ ]:
# Let's try 3D!

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()